In [1]:
!pip install transformers datasets evaluate rouge_score sentencepiece accelerate -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [2]:
# ============================================
# CELL 2: Upload All 3 Dataset Files
# ============================================
from google.colab import files

print("Please select all 3 JSON files together (Ctrl/Cmd + click to multi-select)")
uploaded = files.upload()

# List what got uploaded
print("\nUploaded files:")
for fname in uploaded.keys():
    print(" -", fname)

Please select all 3 JSON files together (Ctrl/Cmd + click to multi-select)


Saving news_dataset.csv to news_dataset.csv
Saving news_dataset_3000.csv to news_dataset_3000.csv
Saving news_dataset_extra.csv to news_dataset_extra.csv

Uploaded files:
 - news_dataset.csv
 - news_dataset_3000.csv
 - news_dataset_extra.csv


In [4]:
# ============================================
# CELL 3: Load Each CSV & Inspect Structure
# ============================================
import pandas as pd

file_names = list(uploaded.keys())  # ['news_dataset.csv', 'news_dataset_3000.csv', 'news_dataset_extra.csv']
dataframes = {}

for fname in file_names:
    temp_df = pd.read_csv(fname)
    dataframes[fname] = temp_df

    print(f"\n=== {fname} ===")
    print("Shape:", temp_df.shape)
    print("Columns:", temp_df.columns.tolist())


=== news_dataset.csv ===
Shape: (571, 5)
Columns: ['title', 'content', 'source', 'link', 'word_count']

=== news_dataset_3000.csv ===
Shape: (640, 3)
Columns: ['title', 'content', 'source']

=== news_dataset_extra.csv ===
Shape: (861, 4)
Columns: ['title', 'content', 'source', 'word_count']


In [5]:
# ============================================
# CELL 4: Preview One Sample Row From Each File
# ============================================
for fname, d in dataframes.items():
    print(f"\n=== Sample row from {fname} ===")
    print(d.iloc[0])
    print("-" * 60)


=== Sample row from news_dataset.csv ===
title         Silicon Valley Bank collapse renews calls to a...
content       When customers at Silicon Valley Bank rushed t...
source                                                CNN_World
link          https://www.cnn.com/2023/04/13/business/silico...
word_count                                                   36
Name: 0, dtype: object
------------------------------------------------------------

=== Sample row from news_dataset_3000.csv ===
title      Fulton County, Georgia, jail leadership resign...
content    Three officials at the Fulton County, Georgia,...
source                                                CNN_US
Name: 0, dtype: object
------------------------------------------------------------

=== Sample row from news_dataset_extra.csv ===
title         Fulton County, Georgia, jail leadership resign...
content       Three officials at the Fulton County, Georgia,...
source                                                   CNN_US


In [6]:
# ============================================
# CELL 5: Merge All 3 Datasets into One
# ============================================

# Add a column to track which file each row came from (useful for debugging/audit)
for fname, d in dataframes.items():
    d['source_file'] = fname

# Concatenate all 3 — pandas will auto-align columns and fill missing ones (link, word_count) with NaN
merged_df = pd.concat(dataframes.values(), ignore_index=True, sort=False)

print("Merged shape (before dedup):", merged_df.shape)
print("Columns:", merged_df.columns.tolist())
merged_df.head()

Merged shape (before dedup): (2072, 6)
Columns: ['title', 'content', 'source', 'link', 'word_count', 'source_file']


,title,content,source,link,word_count,source_file
0,Silicon Valley Bank collapse renews calls to a...,When customers at Silicon Valley Bank rushed t...,CNN_World,https://www.cnn.com/2023/04/13/business/silico...,36.0,news_dataset.csv
1,Big-box stores could help slash emissions and ...,As the US attempts to wean itself off its heav...,CNN_World,https://www.cnn.com/2022/03/20/us/solar-power-...,34.0,news_dataset.csv
2,Look of the Week: Blackpink headline Coachella...,Bringing the second day of this year's Coachel...,CNN_World,https://www.cnn.com/style/article/blackpink-co...,64.0,news_dataset.csv
3,How Playboy cut ties with Hugh Hefner to creat...,Hugh Hefner launched Playboy Magazine 70 years...,CNN_World,https://www.cnn.com/style/article/playboy-the-...,31.0,news_dataset.csv
4,Michael Jordan's 1998 NBA Finals sneakers sell...,"In 1998, Michael Jordan laced up a pair of his...",CNN_World,https://www.cnn.com/style/article/michael-jord...,84.0,news_dataset.csv


In [8]:
# ============================================
# CELL 6a: Verify Duplicates Before Dropping (Diagnostic)
# ============================================

# Re-create merged_df fresh (without dropping yet) so we can inspect duplicates first
merged_df_check = pd.concat(dataframes.values(), ignore_index=True, sort=False)
print("Shape before any dedup:", merged_df_check.shape)

# Find rows that are duplicated based on title+content (mark ALL occurrences, not just extras)
dup_mask = merged_df_check.duplicated(subset=['title', 'content'], keep=False)
duplicates_df = merged_df_check[dup_mask].sort_values(by='title')

print("Total rows involved in duplication:", duplicates_df.shape[0])
print("Unique duplicate groups (title+content):", duplicates_df.duplicated(subset=['title','content']).sum())

Shape before any dedup: (2072, 6)
Total rows involved in duplication: 1484
Unique duplicate groups (title+content): 857


In [9]:
# ============================================
# CELL 6b: Manually Inspect a Few Duplicate Groups
# ============================================
# Group by title+content and show a few groups side-by-side to confirm they're TRUE duplicates
# (not just similar titles with different content, or truncated content)

grouped = duplicates_df.groupby(['title', 'content'])

count = 0
for (title, content), group in grouped:
    if count >= 3:  # show first 3 groups only
        break
    print(f"\n{'='*80}")
    print(f"TITLE: {title[:100]}")
    print(f"Number of copies: {len(group)}")
    print(f"Source files: {group['source_file'].tolist()}")
    print(f"Content length (chars): {len(str(content))}")
    print(f"Content preview: {str(content)[:200]}")
    count += 1


TITLE: "Sabotage": Experts, lawmakers blast RFK Jr. for destroying healthcare research
Number of copies: 2
Source files: ['news_dataset_3000.csv', 'news_dataset_extra.csv']
Content length (chars): 967
Content preview: The federal agency tasked with studying ways to improve America's outstandingly poor healthcare system is "on the brink," experts warn. The Trump administration has cut its staff by 75 percent, cancel

TITLE: $200,000 worth of supplies donated to PB schools
Number of copies: 3
Source files: ['news_dataset_3000.csv', 'news_dataset_extra.csv', 'news_dataset.csv']
Content length (chars): 2820
Content preview: NewsEducationActionsFacebookTweetEmail$200,000 worth of supplies distributed to Palm Beach County schools during giveaway event'This goes a long way to make sure kids have the supplies they need to st

TITLE: 'A definitive backslide.' Inside fashion's worrying runway trend
Number of copies: 3
Source files: ['news_dataset_extra.csv', 'news_dataset.csv', 'news_dataset_30

In [10]:
# ============================================
# CELL 6c: Check if Duplicates Are EXACT or Near-Duplicate
# ============================================
# Check content length variation WITHIN duplicate groups
# If content lengths differ significantly within a "duplicate" group,
# our matching might be too loose (or too strict) — worth knowing

content_len_variation = duplicates_df.groupby(['title'])['content'].apply(
    lambda x: x.str.len().nunique()
)
print("Duplicate groups where content length VARIES (potential false-duplicate risk):")
print((content_len_variation > 1).sum(), "out of", content_len_variation.shape[0], "groups")

Duplicate groups where content length VARIES (potential false-duplicate risk):
0 out of 626 groups


In [11]:
# ============================================
# CELL 6: Finalize Deduplication (Confirmed Safe)
# ============================================
before = merged_df_check.shape[0]
merged_df = merged_df_check.drop_duplicates(subset=['title', 'content'], keep='first').reset_index(drop=True)
after = merged_df.shape[0]

print(f"Removed {before - after} confirmed exact duplicates")
print("Final clean merged shape:", merged_df.shape)

# Drop the diagnostic dataframe now that we've confirmed correctness
del merged_df_check, duplicates_df

Removed 857 confirmed exact duplicates
Final clean merged shape: (1215, 6)


In [12]:
# ============================================
# CELL 7: Basic Data Quality Check on Final Merged Dataset
# ============================================
print("Missing values per column:\n", merged_df.isnull().sum())
print("\nSource distribution:\n", merged_df['source'].value_counts())
print("\nRows per source_file:\n", merged_df['source_file'].value_counts())

Missing values per column:
 title           11
content          0
source           0
link           649
word_count     414
source_file      0
dtype: int64

Source distribution:
 source
Independent                50
Wired                      50
CNBC                       49
Guardian_World             45
FoxNews                    44
BBC_Business               43
Google_News                36
BBC_World                  33
CNN_World                  30
CNN_US                     29
Guardian_US                28
NYT_World                  28
Guardian_Tech              27
NYT_US                     26
Guardian_Business          25
Bloomberg                  23
NYT_Business               20
TechCrunch                 20
The_Diplomat               20
The_Atlantic               20
Harvard_Gazette            20
Space_com                  20
BuzzFeed_News              20
SCMP                       20
Science_Daily              20
TheVerge                   20
Slate                      20
Busin

In [13]:
# ============================================
# CELL 8: Content Length Analysis (word count)
# ============================================
merged_df['content_word_count'] = merged_df['content'].astype(str).apply(lambda x: len(x.split()))

print(merged_df['content_word_count'].describe())

# Flag very short articles (likely scraping artifacts / boilerplate-only content)
short_articles = (merged_df['content_word_count'] < 30).sum()
print(f"\nArticles with < 30 words (likely junk/incomplete): {short_articles}")

count     1215.000000
mean       767.125926
std       1292.231169
min         25.000000
25%         90.500000
50%        436.000000
75%        931.500000
max      17389.000000
Name: content_word_count, dtype: float64

Articles with < 30 words (likely junk/incomplete): 1


In [14]:
# ============================================
# CELL 9: Clean Dataset — Drop Junk Rows & Truncate Prep
# ============================================
# Remove articles too short to summarize meaningfully
clean_df = merged_df[merged_df['content_word_count'] >= 30].reset_index(drop=True)

# Remove common scraped boilerplate patterns (nav menus, share buttons, etc.)
import re
def clean_text(text):
    text = re.sub(r'FacebookTweetEmail(Link)?(Threads)?(Link Copied!)?(Follow)?', '', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

clean_df['content'] = clean_df['content'].apply(clean_text)
print("Final rows ready for summarization:", clean_df.shape[0])

Final rows ready for summarization: 1214


In [16]:
# ============================================
# CELL 10 (FIXED): Manually Load HF Text-to-Text Model
# ============================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded on: {device}")

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model loaded on: cuda


In [17]:
# ============================================
# CELL 10b: Manual Summarization Function (core T2T logic)
# ============================================
def generate_summary(text, max_input_tokens=1024, max_output_tokens=60, min_output_tokens=15):
    # Tokenize input text -> model's expected numeric format
    inputs = tokenizer(
        text,
        max_length=max_input_tokens,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # Generate output token IDs (this is the "text-to-text" generation step)
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_output_tokens,
        min_length=min_output_tokens,
        num_beams=4,          # beam search for better quality
        length_penalty=2.0,
        early_stopping=True
    )

    # Decode token IDs back into readable text
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# Quick test on 1 article
test_summary = generate_summary(clean_df.loc[0, 'content'])
print("TITLE:", clean_df.loc[0, 'title'])
print("GENERATED SUMMARY:", test_summary)

TITLE: Silicon Valley Bank collapse renews calls to address disparities impacting entrepreneurs of color
GENERATED SUMMARY: Venture capitalist Arlan Hamilton stepped in to help some founders of color who panicked about losing access to payroll funds.


In [18]:
# ============================================
# CELL 11: Run Summarization on Full Dataset (Batch)
# ============================================
from tqdm import tqdm

summaries = []

for text in tqdm(clean_df['content'].tolist()):
    try:
        summary = generate_summary(text)
    except Exception as e:
        summary = ""  # fallback for any malformed rows
    summaries.append(summary)

clean_df['generated_summary'] = summaries
print("Done. Sample check:")
clean_df[['title', 'generated_summary']].head(5)

100%|██████████| 1214/1214 [16:45<00:00,  1.21it/s]

Done. Sample check:


,title,generated_summary
0,Silicon Valley Bank collapse renews calls to a...,Venture capitalist Arlan Hamilton stepped in t...
1,Big-box stores could help slash emissions and ...,As the US attempts to wean itself off its heav...
2,Look of the Week: Blackpink headline Coachella...,Blackpink became the first Asian act to ever h...
3,How Playboy cut ties with Hugh Hefner to creat...,Hugh Hefner launched Playboy Magazine 70 years...
4,Michael Jordan's 1998 NBA Finals sneakers sell...,The game-winning sneakers sold for $2.2 millio...


In [19]:
# ============================================
# CELL 12: Save Final Output
# ============================================
output_cols = ['title', 'content', 'generated_summary', 'source']
final_df = clean_df[output_cols]

final_df.to_csv('news_summaries_final.csv', index=False)

print("Saved: news_summaries_final.csv")
print("Total articles summarized:", final_df.shape[0])
print("\nEmpty/failed summaries:", (final_df['generated_summary'] == "").sum())

Saved: news_summaries_final.csv
Total articles summarized: 1214

Empty/failed summaries: 0


In [20]:
# ============================================
# CELL 13: Download the Final CSV
# ============================================
from google.colab import files
files.download('news_summaries_final.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>